In [9]:
import pandas as pd
import os
wom_path = '/Users/bpb/repos/webofmicrobes/wom'
data_filename = 'exometabolite_data/nature-2021-sonnenburg_in-vitro-data.txt'
data_filename = os.path.join(wom_path, data_filename)
metadata_filename = data_filename.replace('in-vitro-data.txt', 'in-vitro-metadata.txt')
df = pd.read_csv(data_filename, sep='\t',index_col=0)
df.index.name = 'observation_id'
df.reset_index(inplace=True)
df.fillna(0, inplace=True)
metadata_df = pd.read_csv(metadata_filename, sep='\t',index_col=0)
metadata_df.index.name = 'observation_id'
metadata_df.reset_index(inplace=True)

In [10]:
# Map Nature 2021 log2 fold-change values to WoM assertions
import numpy as np

CONSUMED_THRESHOLD = -2.0
SECRETED_THRESHOLD = 2.0

def wom_assertion_label(val: float) -> str:
    if pd.isna(val):
        return "neutral"
    val = float(val)
    if val <= CONSUMED_THRESHOLD:
        return "consumed"
    if val >= SECRETED_THRESHOLD:
        return "secreted"
    return "neutral"

# Identify identifier columns (non-metabolite metadata)
id_cols = ['observation_id']

# Metabolite columns are all the others
numeric_cols = [c for c in df.columns if c not in id_cols]

wom_assertions_df = df.melt(
    id_vars=id_cols,
    value_vars=numeric_cols,
    var_name="compound_name",
    value_name="assertion_value",
)

wom_assertions_df["assertion"] = wom_assertions_df["assertion_value"].map(wom_assertion_label)

# Optional quick sanity check
print(wom_assertions_df["assertion"].value_counts(dropna=False))
# wom_assertions_df = wom_assertions_df[wom_assertions_df["assertion"] != "neutral"]
print(wom_assertions_df.sample(5).to_string())

assertion
neutral     386139
consumed     14301
secreted     11817
Name: count, dtype: int64
        observation_id                         compound_name  assertion_value assertion
273879             147                               ASP-PHE        -0.588976   neutral
193718            1021                             STACHYOSE         0.000000   neutral
313346            1372  GUANOSINE 3',5'-CYCLIC MONOPHOSPHATE         0.000000   neutral
136229            1536                 4-HYDROXYBENZALDEHYDE         0.000000   neutral
311875             510                             GUANOSINE        -4.753362  consumed


In [11]:
import sys
sys.path.insert(0, wom_path)

from loaders.compound_mapper import map_compound_names_batch

# Build a mapping of original compound names → expanded names (faster on unique compounds)
unique_compounds = wom_assertions_df['compound_name'].unique()
print(f"[1/4] Unique compounds before mapping: {len(unique_compounds)}")

# Batch map all unique compounds at once (one call)
print(f"[2/4] Batch mapping all {len(unique_compounds)} unique compounds...")
all_mappings = map_compound_names_batch(list(unique_compounds))

compound_expansion_map = {}
compounds_needing_split = []

# First pass: identify which compounds map well vs. need splitting
for compound_name in unique_compounds:
    compound_name = compound_name.strip()
    full_result = all_mappings.get(compound_name)
    
    if full_result and full_result.match_score >= 0.80:
        # Good match, keep as-is
        compound_expansion_map[compound_name] = [compound_name]
    else:
        # Potential grouping, needs splitting
        compounds_needing_split.append(compound_name)

good_matches = len(unique_compounds) - len(compounds_needing_split)
print(f"      ✓ {good_matches} compounds with good match (>= 0.80)")
print(f"      → {len(compounds_needing_split)} compounds to test for splitting")

# Second pass: split and map only those that need it
split_count = 0
for i, compound_name in enumerate(compounds_needing_split):
    if (i + 1) % max(1, len(compounds_needing_split) // 10) == 0 or i == 0:
        print(f"      Processing splits... {i+1}/{len(compounds_needing_split)}")
    
    parts = [p.strip() for p in compound_name.split(",")]
    
    if len(parts) == 1:
        # No actual commas, keep original
        compound_expansion_map[compound_name] = [compound_name]
    else:
        # Multiple parts—map them all in one batch
        part_mappings = map_compound_names_batch(parts)
        successful_parts = [p for p in parts if part_mappings.get(p) and part_mappings[p].match_score >= 0.70]
        
        if len(successful_parts) >= len(parts) * 0.75:
            compound_expansion_map[compound_name] = successful_parts
            split_count += 1
        else:
            compound_expansion_map[compound_name] = [compound_name]

print(f"      ✓ {split_count} compounds successfully split into individual metabolites")

# Now apply the expansion map to the dataframe
print(f"[3/4] Expanding rows based on compound mapping...")
expanded_rows = []

for original_name, expanded_names in compound_expansion_map.items():
    # Get all rows with this original compound name
    rows_with_compound = wom_assertions_df[wom_assertions_df['compound_name'] == original_name]
    
    # Create a new row for each expanded name
    for expanded_name in expanded_names:
        for _, row in rows_with_compound.iterrows():
            new_row = row.copy()
            new_row['compound_name'] = expanded_name
            if expanded_name != original_name:
                new_row['original_grouped_name'] = original_name
            expanded_rows.append(new_row)

wom_assertions_expanded_df = pd.DataFrame(expanded_rows).reset_index(drop=True)

print(f"[4/4] Summary:")
print(f"      Original unique compounds: {len(unique_compounds)}")
print(f"      Original rows: {len(wom_assertions_df):,}")
print(f"      After expansion: {len(wom_assertions_expanded_df):,}")
print(f"      Row expansion factor: {len(wom_assertions_expanded_df) / len(wom_assertions_df):.2f}x")

print(f"\nSample of expanded data:")
sample = wom_assertions_expanded_df[wom_assertions_expanded_df['compound_name'].str.len() < 30]
print(sample.head(10))

[1/4] Unique compounds before mapping: 393
[2/4] Batch mapping all 393 unique compounds...
      ✓ 324 compounds with good match (>= 0.80)
      → 69 compounds to test for splitting
      Processing splits... 1/69
      Processing splits... 6/69
      Processing splits... 12/69
      Processing splits... 18/69
      Processing splits... 24/69
      Processing splits... 30/69
      Processing splits... 36/69
      Processing splits... 42/69
      Processing splits... 48/69
      Processing splits... 54/69
      Processing splits... 60/69
      Processing splits... 66/69
      ✓ 47 compounds successfully split into individual metabolites
[3/4] Expanding rows based on compound mapping...
[4/4] Summary:
      Original unique compounds: 393
      Original rows: 412,257
      After expansion: 502,471
      Row expansion factor: 1.22x

Sample of expanded data:
   observation_id               compound_name  assertion_value assertion  \
0               3  2-HYDROXYPHENYLACETIC ACID         0.00

In [12]:
wom_assertions_expanded_df[wom_assertions_expanded_df['observation_id']==1599]

,observation_id,compound_name,assertion_value,assertion,original_grouped_name
971,1599,2-HYDROXYPHENYLACETIC ACID,0.000000,neutral,NaN
2020,1599,3-HYDROXYBUTYRIC ACID,0.000000,neutral,NaN
3069,1599,2-HYDROXY-4-METHYLPENTANOIC ACID,0.000000,neutral,NaN
4118,1599,"1,3-DIAMINOPROPANE",12.868762,secreted,NaN
5167,1599,TRIMETHYLAMINE N-OXIDE,0.000000,neutral,NaN
...,...,...,...,...,...
498197,1599,LEUCINE,-2.849206,consumed,"LEUCINE, NORLEUCINE"
499246,1599,NORLEUCINE,-2.849206,consumed,"LEUCINE, NORLEUCINE"
500295,1599,SYRINGIC ACID,0.000000,neutral,NaN
501344,1599,ERYTHRITOL,4.888224,secreted,"ERYTHRITOL, THREITOL"


In [21]:
phylogeny_df = pd.read_csv('/Users/bpb/repos/webofmicrobes/wom/exometabolite_data/phylogeny.tsv', sep='\t')
print(phylogeny_df.head().to_string())
# print(metadata_df.head().to_string())

    Kingdom          Phylum               Class            Order          Family       Genus              Isolate Closest relative in NCBI 16S database  Similarity (%)
0  Bacteria  Pseudomonadota  Betaproteobacteria  Burkholderiales  Comamonadaceae  Acidovorax          GW101-11A03     Acidovorax delafieldii strain 133            98.0
1  Bacteria  Pseudomonadota  Betaproteobacteria  Burkholderiales  Comamonadaceae  Acidovorax          GW101-20D03           Acidovorax soli strain BL21            99.0
2  Bacteria  Pseudomonadota  Betaproteobacteria  Burkholderiales  Comamonadaceae  Acidovorax           GW101-3H11           Acidovorax soli strain BL21            99.0
3  Bacteria  Pseudomonadota  Betaproteobacteria  Burkholderiales  Comamonadaceae  Acidovorax             GW456-A8                 Acidovorax delafildii            98.0
4  Bacteria  Pseudomonadota  Betaproteobacteria  Burkholderiales  Comamonadaceae  Acidovorax  GW458-11-26-14-TSB4                        Acidovorax sp.         

In [22]:
# Reformat metadata_df to match phylogeny_df structure
idx = metadata_df['sample_type']=='supernatant'
metadata_reformatted = metadata_df[idx].copy()

# Step 1: Rename taxonomic columns to match phylogeny_df (capitalize first letter)
taxonomy_column_mapping = {
    'kingdom': 'Kingdom',
    'phylum': 'Phylum',
    'class': 'Class',
    'order': 'Order',
    'family': 'Family',
    'genus': 'Genus',
}

for old_name, new_name in taxonomy_column_mapping.items():
    if old_name in metadata_reformatted.columns:
        metadata_reformatted = metadata_reformatted.rename(columns={old_name: new_name})

# Step 2: Create pseudo strain IDs for observations lacking one
# Use strain if available, otherwise create obs_<observation_id>
def create_strain_id(row):
    if pd.notna(row.get('strain')):
        return str(row['strain'])
    else:
        return f"unk{row['observation_id']}"

metadata_reformatted['strain_id'] = metadata_reformatted.apply(create_strain_id, axis=1)

# Step 3: Concatenate strain ID with media column using delimiter
# Format: strain_id__media (using double underscore as delimiter)
def create_isolate_id(row):
    strain_id = row['strain_id']
    media = row.get('media', 'unknown')
    if pd.notna(media):
        return f"{strain_id}.{media}"
    else:
        return strain_id

metadata_reformatted['Isolate'] = metadata_reformatted.apply(create_isolate_id, axis=1)

# Keep observation_id and all renamed taxonomy columns plus Isolate
columns_to_keep = ['observation_id', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Isolate']
# Add any other columns that exist
for col in columns_to_keep:
    if col not in metadata_reformatted.columns:
        print(f"Warning: {col} not found in metadata")

# Select final columns (keeping only those that exist)
final_columns = [col for col in columns_to_keep if col in metadata_reformatted.columns]
metadata_reformatted = metadata_reformatted[final_columns]
new_cols = ['Closest relative in NCBI 16S database', 'Similarity (%)']
metadata_reformatted[new_cols[0]] = 'unknown'
metadata_reformatted[new_cols[1]] = 0
print(f"Reformatted metadata shape: {metadata_reformatted.shape}")
print(f"\nColumns: {list(metadata_reformatted.columns)}")
print(f"\nSample rows:")
print(metadata_reformatted.head())

Reformatted metadata shape: (931, 10)

Columns: ['observation_id', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Isolate', 'Closest relative in NCBI 16S database', 'Similarity (%)']

Sample rows:
   observation_id   Kingdom         Phylum        Class          Order  \
0               3  Bacteria     Firmicutes   Clostridia  Clostridiales   
4              13  Bacteria  Bacteroidetes  Bacteroidia  Bacteroidales   
5              14  Bacteria     Firmicutes   Clostridia  Clostridiales   
6              15  Bacteria     Firmicutes   Clostridia  Clostridiales   
7              16  Bacteria     Firmicutes   Clostridia  Clostridiales   

            Family         Genus       Isolate  \
0  Lachnospiraceae   Coprococcus   HPP0074.pyg   
4   Bacteroidaceae   Bacteroides      unk13.mm   
5  Lachnospiraceae  Anaerostipes  3_2_56FAA.mm   
6  Lachnospiraceae  Anaerostipes  3_2_56FAA.mm   
7  Lachnospiraceae  Anaerostipes  3_2_56FAA.mm   

  Closest relative in NCBI 16S database  Simi

In [35]:
cols = ['observation_id', 'Isolate']
wom_assertions_formated = pd.merge(
    wom_assertions_expanded_df,
    metadata_reformatted[cols],
    on='observation_id',
    how='left')
wom_assertions_formated = wom_assertions_formated[pd.notna(wom_assertions_formated['Isolate'])]
wom_assertions_formated.drop(columns=['observation_id'], inplace=True)
wom_assertions_formated = pd.pivot_table(
    wom_assertions_formated,
    index='Isolate',
    columns='compound_name',
    values='assertion_value',
    # aggfunc=lambda x: x.iloc[0] if len(x) > 0 else 'neutral'
).reset_index()
# save to tsv do not print the index name just the values
wom_assertions_formated.set_index('Isolate', inplace=True)
wom_assertions_formated.index.name = None
wom_assertions_formated.to_csv(os.path.join(wom_path, 'exometabolite_data/nature_2021_wom_format.tsv'), sep='\t', index=True)

In [38]:
metadata_reformatted_wom = metadata_reformatted.copy()
metadata_reformatted_wom.drop(columns=['observation_id'], inplace=True)
metadata_reformatted_wom.drop_duplicates(subset=['Isolate'], inplace=True)
metadata_reformatted_wom = pd.concat([phylogeny_df, metadata_reformatted_wom], ignore_index=True)
metadata_reformatted_wom.to_csv(os.path.join(wom_path, 'exometabolite_data/phylogeny.tsv'), sep='\t', index=False)